# Week 9 Assignment — A Minimum Heap

> After some thought, I decided that a discrete event simulator for an emergency room was too much work for typical assignment. So the assignment for week 9 has been scaled back to the folllowing material.

**Reading:** [Heaps](./heaps.ipynb)

## The task

Last week you built a queue and a stack on top of a list. This week the
structure underneath is a list again, but the rule is different: a
**priority queue** always gives back its *smallest* value first, no matter
the order in which values were added. A **minimum heap** is the standard way
to build one.

I have written most of `MinHeap271` for you: `size`, `peek`, `add`, `get` and
`_is_leaf` are complete. You write the two operations that do the real work of
a heap, the ones we traced by hand on Friday:

- **`_bubble_up()`**: restores the heap after `add` puts a value at the end.
- **`_sink_down()`**: restores the heap after `get` moves the last value to
  the root.

**Part A** is the heap itself. **Part B** is a small design problem that uses
your heap: a waiting list in an emergency department, where patients have to
be seen by severity and, within a severity, in a fair order.

The notes in `heaps.ipynb` cover everything you need for Part A. Read them first if you
missed Friday.

## How this notebook works

- The notebook is the **handout**: the explanation, the scaffold and the
  tests. You do not turn it in; you turn in a Python file (see
  *What to submit* at the end).
- Code cells marked **Given** are complete. Read them. Do not change them.
- The class cell marked **TODO** contains two methods for you to complete.
- A cell of plain-`assert` tests follows. Each test prints `ok` or `FAIL` with
  a reason. Re-run the test cell after every change. Some tests fail on the
  untouched scaffold; that is expected.
- Run the notebook top to bottom. If you change a cell above, re-run
  everything below it.

As always, every function and method that returns a value has **exactly one
`return`**: set a result variable to the "didn't work" value first, and
overwrite it only once the guard condition passes.

In [ ]:
# Given -- the test runner used at the end of each part.
def run_tests(prefix: str) -> None:
    """Run every function in this notebook whose name starts with `prefix`."""
    tests = [(name, fn) for name, fn in sorted(globals().items())
             if name.startswith(prefix) and callable(fn)]
    passed = 0
    for name, fn in tests:
        try:
            fn()
            print(f"ok    {name}")
            passed += 1
        except Exception as err:   # AssertionError, TypeError, IndexError...
            print(f"FAIL  {name}: {type(err).__name__}: {err}")
    print(f"\n{passed} of {len(tests)} checks passed")

---

## Part A — `MinHeap271`

### The interface

All four operations work under one invariant: **`get` always returns the
smallest value in the collection.**

| Method | Success | Failure (nothing changes) |
|---|---|---|
| `add(value)` | places `value` in the heap | — (`add` cannot fail) |
| `get()` | removes and returns the smallest value | heap is empty → returns `None` |
| `peek()` | returns the smallest value **without** removing it | heap is empty → returns `None` |
| `size()` | returns the number of values in the heap | — |

### The representation

The heap is a complete binary tree stored in a Python list, `self.storage`,
level by level, left to right. The root is at `storage[0]` and the last leaf
is at `storage[len(storage) - 1]`. Parents and children are found with the
arithmetic from class:

\begin{align}
\text{left}(p) & = 2p+1 \\
\text{right}(p) & = 2p+2 \\
\text{parent}(c) & = \left\lfloor \frac{c-1}{2} \right\rfloor
\end{align}

Unlike last week's queue and stack, this storage **does** grow and shrink:
`add` appends to the end of the list and `get` removes the last element.
Those are the only two places the list changes size.

The **heap property** is: every value is less than or equal to both of its
children. Equivalently, `storage[parent(c)] <= storage[c]` for every position
`c` after the root.

### How `add` and `get` use your two methods

Read the given `add` and `get`, and trace them by hand on a small heap.

- `add` appends the new value at the end. The tree is still complete, but the
  new value may be smaller than its parent. `_bubble_up` fixes that.
- `get` remembers the root (the answer), removes the last value from the list,
  and, if anything is left, puts that value at the root. The tree is still
  complete, but the root may now be larger than its children. `_sink_down`
  fixes that.

Notice that `get` on a heap with exactly one value never calls `_sink_down`.
Why not? What would be in `storage` at that point?

### What you write

- **`_bubble_up()`.** Called by `add`, right after the new value has been
  appended. Track the new value's position, starting at `len - 1`. While it
  is not at the root and it is smaller than its parent, swap it with its
  parent and follow it up.
- **`_sink_down()`.** Called by `get`, right after the last value has been
  moved into the vacant root. Track that value's position, starting at `0`.
  While it is not at a leaf, find its **smaller** child: assume the left
  child, then switch to the right child if the right child exists and is
  smaller. If the value is greater than that smaller child, swap them and
  follow the value down. Otherwise, stop.

Use the given `_parent`, `_left_child`, `_right_child`, `_swap` and
`_is_leaf`. Before you code, trace both methods on paper with the values
`2, 8, 4` followed by `add(1)`, and then with two calls to `get`.

### Rules

- No `import heapq`, and no `sorted()`, `.sort()`, `min()` or `max()` over
  `storage`. The whole point is that the heap finds the minimum by itself.
- Compare values only with `<` and `>`. That way the heap works for any
  values that Python can compare, including tuples such as `(2, "Ana")`.

In [ ]:
# Given + TODO -- complete the two methods marked TODO.
class MinHeap271:
    """A priority queue that always returns its smallest value, stored as a
    complete binary tree in a Python list.

    storage -- the tree, level by level, left to right; the root is storage[0]
    """

    def __init__(self):
        self.storage: list = []

    # ---- Given: index arithmetic and swapping

    def _left_child(self, parent: int) -> int:
        return 2 * parent + 1

    def _right_child(self, parent: int) -> int:
        return 2 * parent + 2

    def _parent(self, child: int) -> int:
        return (child - 1) // 2

    def _swap(self, i: int, j: int) -> None:
        temp = self.storage[i]
        self.storage[i] = self.storage[j]
        self.storage[j] = temp

    def _is_leaf(self, position: int) -> bool:
        """Return True if position has no children in storage. A position has
        a child exactly when its left child is inside the list."""
        leaf = self._left_child(position) >= len(self.storage)
        return leaf

    # ---- TODO: restoring the heap property

    def _bubble_up(self) -> None:
        """Move the last value in storage up until its parent is not greater
        than it, or it reaches the root."""
        # TODO
        pass

    def _sink_down(self) -> None:
        """Move the value at the root down until neither child is smaller
        than it, or it reaches a leaf."""
        # TODO
        pass

    # ---- Given: the priority queue interface

    def size(self) -> int:
        """Return the number of values in the heap."""
        count = len(self.storage)
        return count

    def peek(self):
        """Return the smallest value without removing it; None if empty."""
        smallest = None
        if len(self.storage) > 0:
            smallest = self.storage[0]
        return smallest

    def add(self, value) -> None:
        """Place value in the heap, keeping the minimum heap property."""
        self.storage.append(value)
        self._bubble_up()

    def get(self):
        """Remove and return the smallest value; None if empty."""
        smallest = None
        if len(self.storage) > 0:
            smallest = self.storage[0]
            last = self.storage.pop()          # remove the last leaf
            if len(self.storage) > 0:          # something is left: move it to the root
                self.storage[0] = last
                self._sink_down()
        return smallest

    def __len__(self) -> int:
        return self.size()

    def __str__(self) -> str:
        return f"MinHeap271{self.storage}"

### Tests

`test_heap_property_holds` checks the array itself after every operation:
every parent must be less than or equal to both its children. If your `get`
returns the right values but this test fails, your heap is getting the right
answers by accident.

In [ ]:
# Given -- tests. Run this cell after every change.
import inspect
import random


def heap_property_ok(heap: MinHeap271) -> bool:
    """True if every parent in heap.storage is <= both of its children."""
    a = heap.storage
    ok = True
    for child in range(1, len(a)):
        if a[(child - 1) // 2] > a[child]:
            ok = False
    return ok


def test_heap_empty():
    h = MinHeap271()
    assert h.size() == 0, "a new heap has size 0"
    assert h.peek() is None, "peek on an empty heap returns None"
    assert h.get() is None, "get on an empty heap returns None"
    assert h.size() == 0, "a failed get changes nothing"


def test_heap_is_leaf():
    h = MinHeap271()
    h.storage = [1, 2, 3, 4]          # set directly: tests _is_leaf alone
    assert not h._is_leaf(0), "the root has two children"
    assert not h._is_leaf(1), "position 1 has a left child at 3"
    assert h._is_leaf(2), "position 2 would have children at 5 and 6"
    assert h._is_leaf(3), "position 3 is the last leaf"


def test_heap_single_value():
    h = MinHeap271()
    h.add(42)
    assert h.size() == 1
    assert h.peek() == 42, "peek returns the only value"
    assert h.size() == 1, "peek does not remove"
    assert h.get() == 42, "get returns the only value"
    assert h.size() == 0, "and the heap is empty again"
    assert h.get() is None


def test_heap_bubble_up_class_example():
    # From class: 2, 8, 4 is a minimum heap; adding 1 floats to the root.
    h = MinHeap271()
    for v in [2, 8, 4]:
        h.add(v)
    h.add(1)
    assert h.storage == [1, 2, 4, 8], f"expected [1, 2, 4, 8], got {h.storage}"


def test_heap_sink_down_swaps_with_smaller_child():
    # 3 moves to the root and must swap with 1 (the smaller child), not 2.
    h = MinHeap271()
    h.storage = [0, 2, 1, 3]          # a valid minimum heap
    assert h.get() == 0
    assert h.storage == [1, 2, 3], f"expected [1, 2, 3], got {h.storage}"


def test_heap_ascending_and_descending_adds():
    for values in ([1, 2, 3, 4, 5, 6, 7], [7, 6, 5, 4, 3, 2, 1]):
        h = MinHeap271()
        for v in values:
            h.add(v)
        out = [h.get() for _ in values]
        assert out == [1, 2, 3, 4, 5, 6, 7], f"added {values}, got {out}"


def test_heap_duplicates():
    h = MinHeap271()
    for v in [2, 1, 2, 1, 2]:
        h.add(v)
    out = [h.get() for _ in range(5)]
    assert out == [1, 1, 2, 2, 2], f"got {out}"


def test_heap_property_holds():
    random.seed(271)
    h = MinHeap271()
    for _ in range(300):
        if random.random() < 0.6:
            h.add(random.randint(-50, 50))
        else:
            h.get()
        assert heap_property_ok(h), f"heap property broken: {h.storage}"


def test_heap_random_matches_sorted():
    random.seed(9)
    for n in [2, 3, 10, 31, 32, 33, 200]:
        values = [random.randint(0, 100) for _ in range(n)]
        h = MinHeap271()
        for v in values:
            h.add(v)
        assert h.size() == n
        out = [h.get() for _ in range(n)]
        assert out == sorted(values), f"n={n}: values came out out of order"
        assert h.get() is None


def test_heap_tuples():
    # Python compares tuples element by element, so the first element decides
    # and ties go to the second. The heap does not care what it stores.
    h = MinHeap271()
    for item in [(3, "c"), (2, "b"), (1, "a"), (2, "a"), (1, "b")]:
        h.add(item)
    out = [h.get() for _ in range(5)]
    assert out == [(1, "a"), (1, "b"), (2, "a"), (2, "b"), (3, "c")], f"got {out}"


def test_heap_no_shortcuts():
    # Read each method's source. (inspect.getsource on a class defined in a
    # notebook can fail; on its functions it works.)
    src = "".join(inspect.getsource(member) for member in vars(MinHeap271).values()
                  if inspect.isfunction(member))
    for banned in ["heapq", "sorted(", ".sort(", "min(", "max("]:
        assert banned not in src, f"MinHeap271 may not use {banned}"


run_tests("test_heap")

### Using the heap

Once the tests pass, the cell below uses your heap to sort a list: add every
value, then call `get` until the heap is empty. The values come out in order
although you never compared them to each other directly. It is called
*heap sort*.

In [ ]:
# Given -- heap sort with your heap.
random.seed(271)
values = [random.randint(0, 99) for _ in range(15)]
heap = MinHeap271()
for v in values:
    heap.add(v)
print("heap storage after the adds:", heap.storage)
in_order = [heap.get() for _ in range(heap.size())]
print("values:   ", values)
print("in order: ", in_order)

### Part B: who is seen next?

A hospital's emergency department uses a minimum heap as its waiting list.
Every patient who arrives is given a **severity**: `1` is life-threatening,
`2` is severe and `3` is not life-threatening. The smaller the number, the
sooner the patient must be seen, which is what a minimum heap returns first.

The problem: several patients with the same severity may be waiting. The
heap, as written, cannot tell which of them should be seen first.

**The rule we want:**

1. A patient with a smaller severity is always called in before a patient
   with a larger severity.
2. Among patients with the same severity, the patient who **arrived first**
   is called in first.

**What you have and what you may not change:**

- Your `MinHeap271` from Part A. You may not modify the class for Part B. It
  stores whatever you `add` to it, compares values with `<` and `>` and
  nothing else, and knows nothing about patients or time.
- Patients arrive one at a time. The arrival number of a patient is `0` for
  the first patient to arrive, `1` for the second, and so on, over all
  severities.

**What you write.** Patients are **objects**, and you design them.  By design I mean: describe the basic fields and methods of the
class and say what each one is for. You do not need to write the code. How you
build it is up to you; the only fixed parts are:

- `Patient(severity)` creates a patient.
- A patient object has enough structure to help the min heap decide who's next from a group of patients with same severity.
- How that structure is manifested, is up to you. You may give the class any other attributes and methods you find useful.





## What to submit

Turn in **one Python file**, `heap271.py`, containing your `MinHeap271` class
with `_bubble_up` and `_sink_down` completed (the other methods stay as
given), your answers to the questions (including the description of the
`Patient` class for Part B) as comments at the top, and nothing else that runs
automatically. Your file must pass the Part A tests above.
